# Train on Kaggle

Attach competition data and the [wheelhouse Dataset](https://www.kaggle.com/datasets/denysmykhailov/birdclef-2026-wheelhouse).
Choose GPU and Internet ON. For two devices, choose T4 ×2.

**Add the Kaggle secret `WANDB_API_KEY` and give this notebook access to it.**
`WANDB_OFFLINE=False` is the default: it sends logs to W&B. Set it to `True`
only if you want local logs and no key.

Edit the settings below. Batch size and worker count are per GPU. The bundle has
saved folds and pretrained weights. Use Save Version → Save & Run All to train
in a clean session. Download the run ZIP from Output when it ends.
See [remote setup](../../docs/remote.md).

In [1]:
from pathlib import Path

wheel = next(
    path for path in Path("/kaggle/input").rglob("birdclef*.whl")
    if (path.parent / "data_prep" / "manifest.json").is_file()
)
bundle = wheel.parent
%pip install --find-links "{bundle}" -r "{bundle}/requirements.txt" -q
%pip install --force-reinstall --no-deps "{wheel}" -q


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires opentelemetry-api<=1.42.1,>=1.39, but you have opentelemetry-api 1.45.1 which is incompatible.
google-adk 2.7.1 requires opentelemetry-sdk<=1.42.1,>=1.39, but you have opentelemetry-sdk 1.45.1 which is incompatible.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import os

from birdclef.config import RuntimeConfig, TrainConfig, get_config
from birdclef.models import AudioExperimentConfig, AudioModelConfig
from birdclef.runtime import apply_environment_defaults, get_secret

apply_environment_defaults("kaggle")
os.environ["TORCH_HOME"] = str(bundle / "torch")

RUN_NAME = "audio-cnn/partial-unfreeze"
SMOKE_RUN = False
WANDB_OFFLINE = False  # True to log locally and sync later with wandb sync.
CPU_THREADS = 1

os.environ["OMP_NUM_THREADS"] = str(CPU_THREADS)
os.environ["MKL_NUM_THREADS"] = str(CPU_THREADS)
os.environ["OPENBLAS_NUM_THREADS"] = str(CPU_THREADS)

config = get_config(
    AudioExperimentConfig,
    fold=0,
    smoke=SMOKE_RUN,
    model=AudioModelConfig(
        pretrained=True,
        encoder_lr=3e-5,
        head_lr=3e-4,
        unfreeze_epoch=1,
        unfreeze_last_n=2,  # None = all encoder
    ),
    tracking={"offline": WANDB_OFFLINE},
    runtime=RuntimeConfig(accelerator="gpu", devices=2, precision="16-mixed"),
    loader={
        "batch_size": 4 if SMOKE_RUN else 64,  # Per GPU; effective batch = 128.
        "val_batch_size": 128,
        "predict_batch_size": 128,
        "num_workers": 2,  # Per GPU; four workers on a T4 x2 instance.
        "pin_memory": True,
        "seed": 42,
    },
    train=TrainConfig(
        max_epochs=1 if SMOKE_RUN else 6,
        checkpoint_metric="val/auc",
        checkpoint_mode="max",
        early_stopping_patience=3,
        log_every_n_steps=20,
    ),
)
RUN_DIR = config.data.output_dir / config.run_name(RUN_NAME)
print(f"Run directory: {RUN_DIR}")

                           AudioExperimentConfig                            
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Setting                       ┃ Value                                    ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ app_name                      │ birdclef                                 │
│ logging.directory             │ /kaggle/working/logs                     │
│ logging.level                 │ INFO                                     │
│ logging.formatter             │ ColoredFormatter                         │
│ logging.date_format           │ %d-%m-%Y %H:%M:%S                        │
│ logging.full_color            │ True                                     │
│ logging.include_function      │ True                                     │
│ logging.success_level         │ 69                                       │
├───────────────────────────────┼──────────────────────────────────────────┤
│ data.competition_dir          │ /kaggle/input/competitions/birdclef-2026 │
│ data.output_dir               │ /kaggle/working                          │
├───────────────────────────────┼──────────────────────────────────────────┤
│ split.validation_fraction     │ 0.2                                      │
│ split.seed                    │ 42                                       │
│ split.group_column            │ None                                     │
│ split.stratify                │ False                                    │
├───────────────────────────────┼──────────────────────────────────────────┤
│ tracking.project              │ birdclef-2026                            │
│ tracking.entity               │ None                                     │
│ tracking.offline              │ False                                    │
│ tracking.api_key              │ None                                     │
├───────────────────────────────┼──────────────────────────────────────────┤
│ loader.batch_size             │ 64                                       │
│ loader.val_batch_size         │ 128                                      │
│ loader.predict_batch_size     │ 128                                      │
│ loader.num_workers            │ 2                                        │
│ loader.pin_memory             │ True                                     │
│ loader.seed                   │ 42                                       │
│ loader.soundscape_probability │ 0.1                                      │
├───────────────────────────────┼──────────────────────────────────────────┤
│ lr_scheduler.name             │ none                                     │
│ lr_scheduler.t_max            │ 10                                       │
│ lr_scheduler.min_lr           │ 0.0                                      │
│ lr_scheduler.factor           │ 0.1                                      │
│ lr_scheduler.patience         │ 2                                        │
│ lr_scheduler.monitor          │ val/loss                                 │
│ lr_scheduler.mode             │ min                                      │
├───────────────────────────────┼──────────────────────────────────────────┤
│ train.max_epochs              │ 6                                        │
│ train.seed                    │ 42                                       │
│ train.checkpoint_metric       │ val/auc                                  │
│ train.checkpoint_mode         │ max                                      │
│ train.checkpoint_top_k        │ 1                                        │
│ train.early_stopping_patience │ 3                                        │
│ train.accumulate_grad_batches │ 1                                        │
│ train.gradient_clip_val       │ 1.0                                      │
│ train.log_every_n_steps       │ 20                                       │
├───────────────────────────────┼────────────────────────────────────────

Run directory: /kaggle/working/audio-cnn/partial-unfreeze


In [ ]:
if not config.tracking.offline:
    os.environ["WANDB_API_KEY"] = get_secret("WANDB_API_KEY", environment="kaggle") or ""

PREPARED = bundle / "data_prep"
config_path = config.data.output_dir / "launch-config.json"
config_path.parent.mkdir(parents=True, exist_ok=True)
config_path.write_text(json.dumps(config.safe_dump(), indent=2), encoding="utf-8")


1535

In [4]:
!python -u -m birdclef.models.classifier.fit --config "{config_path}" --name "{RUN_NAME}" --prepared-dir "{PREPARED}"

28-10-2026 17:45:15  INFO      [models/audio_cnn/fit:68]  Run: audio-cnn/partial-unfreeze; training examples: 35769; validation windows: 149
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.forge.coreweave.com.
wandb: No netrc file found, creating one.
wandb: Appending key for api.forge.coreweave.com to your netrc file: /root/.netrc
wandb: Currently logged in as: dmykhailov (kaggle-casmi) to https://api.forge.coreweave.com. Use `wandb login --relogin` to force relogin
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: ⣾ Waiting for wandb.init()...
wandb: ⣷ setting up run 93f381c09f2b4122b9b

In [5]:
import shutil
from IPython.display import FileLink, display

print((RUN_DIR / "validation-metrics.json").read_text(encoding="utf-8"))
archive = shutil.make_archive(str(config.data.output_dir / "partial-unfreeze"), "zip", RUN_DIR)
display(FileLink(archive))


{
  "auc": 0.8183267849884105,
  "auc_classes": 28,
  "windows": 149,
  "protocol": "file-gkf5/fold-0"
}


/kaggle/working/partial-unfreeze.zip